### Recommendation

This notebook contains a recommendation system that generates intelligent guidance based on the predicted category of Hajj-related messages. The system assigns a risk level, explains the prediction, and provides appropriate recommendations to help users make safer and more informed decisions.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/campaigns_all_years_with_price_category .csv")

print(df.columns.tolist())

['campaign_name', 'campaign_name_en', 'cities', 'cities_en', 'total_pilgrims', 'rating', 'rank', 'rank_en', 'price_range', 'price_range_en', 'services', 'services_en', 'year', 'licensed', 'Price_Category']


In [ ]:

import pandas as pd


# UI budget label → CSV price_range value(s)
BUDGET_MAP = {
    "economy": ["اقتصادي"],
    "medium": ["متوسط"],
    "premium": ["مميز", "فاخر"],
}

# Accommodation preference
ACCOMMODATION_PREF = {
    "near_haram": ["مكة المكرمة", "المدينة المنورة"],
    "medium": ["جدة", "الطائف", "ينبع"],
    "economy": [],
}

# Transportation
TRANSPORT_BONUS = {
    "buses": ["نقل", "مواصلات داخلية"],
    "train": ["نقل", "قطار"],
    "vip": [],
}

# Services
SERVICE_KEYWORDS = {
    "religious_guidance": ["إرشاد", "مرشد"],
    "meals": ["وجبات", "هدي"],
    "support_24h": ["دعم", "مواصلات داخلية"],
}


def recommend_campaigns(
    budget=None,
    accommodation=None,
    transportation=None,
    services=None,
    year=None,
    top_n=6,
    min_rating=3.5
):

    data = df.copy()

    # Latest year if not specified
    years = sorted(data["year"].dropna().unique(), reverse=True)

    if year:
        data = data[data["year"] == year]
    else:
        data = data[data["year"] == years[0]]

    data = data.copy()
    data["score"] = 0

    # Budget score
    if budget in BUDGET_MAP:
        targets = BUDGET_MAP[budget]

        data.loc[
            data["price_range"].isin(targets),
            "score"
        ] += 30

    # Accommodation score
    if accommodation in ACCOMMODATION_PREF:

        preferred = ACCOMMODATION_PREF[accommodation]

        for city in preferred:
            data.loc[
                data["cities"].fillna("").str.contains(city, case=False),
                "score"
            ] += 25

    # Transportation score
    if transportation in TRANSPORT_BONUS:

        for kw in TRANSPORT_BONUS[transportation]:

            data.loc[
                data["services"].fillna("").str.contains(kw, case=False),
                "score"
            ] += 10

        if transportation == "vip":

            data.loc[
                data["price_range"].isin(["فاخر", "مميز"]),
                "score"
            ] += 15

    # Services score
    if services:

        for service in services:

            for kw in SERVICE_KEYWORDS.get(service, []):

                data.loc[
                    data["services"].fillna("").str.contains(kw, case=False),
                    "score"
                ] += 10

    # Rating filter
    data = data[data["rating"] >= min_rating]

    data["score"] += data["rating"] * 8

    # Licensed bonus
    data.loc[
        data["licensed"] == "نعم",
        "score"
    ] += 15

    # Total pilgrims bonus
    max_pilgrims = max(data["total_pilgrims"].max(), 1)

    data["score"] += (
        data["total_pilgrims"].fillna(0)
        / max_pilgrims
    ) * 10

    # Rank bonus
    data.loc[
        data["rank"].fillna("").str.contains("الأفضل"),
        "score"
    ] += 8

    data.loc[
        data["rank"].fillna("").str.contains("موصى"),
        "score"
    ] += 5

    results = (
        data
        .sort_values("score", ascending=False)
        .head(top_n)
    )

    return results[
        [
            "campaign_name",
            "cities",
            "price_range",
            "rating",
            "services",
            "score"
        ]
    ]


# Example
recommend_campaigns(
    budget="premium",
    accommodation="near_haram",
    transportation="vip",
    services=["meals"],
    top_n=5
)


,campaign_name,cities,price_range,rating,services,score
0,شركه المهابه لخدمات حجاج الداخل المحدوده,"الطائف, جدة",فاخر,5.0,سكن + نقل + هدي + وجبات,137.996324
3,شركه الأنهار المتحدة لخدمة حجاج الداخل,"جدة, ينبع",فاخر,4.7,سكن + نقل + هدي + وجبات + مواصلات داخلية,135.600000
1,شركة تفويج المحدودة,"الطائف, المدينة المنورة, جدة, مكة المكرمة",اقتصادي,5.0,سكن + نقل + هدي,130.466912
36,شركة سيف الاسلام لخدمات حجاج الداخل المحدوده,"الطائف, جدة, مكة المكرمة",مميز,4.2,سكن + نقل,128.919853
38,شركة مخيم النور لخدمات حجاج الداخل المحدوده,"المدينة المنورة, جدة, مكة المكرمة",اقتصادي,4.2,سكن + نقل + هدي + وجبات + مواصلات داخلية,128.548529
